<a href="https://colab.research.google.com/github/EmanuelMatiasCinel/Proyecto-Final-Data-Science/blob/main/Grupo_16_Proyecto_Data_Science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grupo 16 - Proyecto Data Science
## Predicción de la dirección anual del precio de acciones a partir de indicadores financiero-contables




 **Objetivo**: construir un modelo de clasificación que, a partir de los indicadores financiero-contables de una empresa en el año t, prediga si su acción sube o baja durante el año t+1. Como análisis complementario, se agrupan las empresas por perfil financiero (clustering) para ver si ciertos perfiles se asocian a mayor probabilidad de suba.
 El modelo no pretende predecir precios con precisión, sino evaluar si los fundamentals contienen alguna señal de variación.

**Fuente de datos**: “200+ Financial Indicators of US stocks (2014-2018)”, Kaggle, usuario Nicolas Carbone (https://www.kaggle.com/datasets/cnic92/200-financial-indicators-of-us-stocks-20142018). Cada archivo anual contiene más de 200 indicadores por empresa, el sector y la variación de precio del año siguiente.

**Variable objetivo**: Clase (1 = la acción sube en el año siguiente, 0 = no sube), derivada de la variación de precio que ya trae el dataset.

# Parte 1: ANALISIS EXPLORATORIO DE LA BASE DE DATOS

## 0. Preparación del Entrono

In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [2]:
import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, RocCurveDisplay, balanced_accuracy_score,
                             classification_report, f1_score, roc_auc_score)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
SEMILLA = 42

Carga de datos y Diagnóstico inicial
##1. Carga y unificación de los 5 años

Se cargaron los 5 archivos (2014_Financial_Data.csv - 2018_Financial_Data.csv) y se guardaron con nombres. Cuántas filas y columnas hay y de qué tipo es cada variable. Lo hacemos solo con una de las 5 tablas para conocer su estructura.

In [3]:
#define los data frames y lee los .csv

df= pd.read_csv("2014_Financial_Data.csv")
df1= pd.read_csv("2015_Financial_Data.csv")
df2= pd.read_csv("2016_Financial_Data.csv")
df3= pd.read_csv("2017_Financial_Data.csv")
df4= pd.read_csv("2018_Financial_Data.csv")

#imprime por consola las primeras 5 filas del DataFrame
print(df.head())

# Imprime info completa del DF
print(df.info())

# Obtengo el nombre de las columnas
df.columns

FileNotFoundError: [Errno 2] No such file or directory: '2014_Financial_Data.csv'

## Corrección columnas y unificación de los 5 años

Detectamos que la primer columna donde se nombra el ticker de la empresa, aparece en blanco, es asi que Renombramos esa columna y agregamos una nueva columna que demarque el Año de la base de datos, llamada `Year`. El resto de los nombres los dejamos tal cual vienen del CSV original (con mayúsculas y espacios) porque todavía los necesitamos así para la selección de columnas del paso 3.


Se crea una sola tabla con los 5 .cvs creando una columna "anio" y nombrando la columna del nombre de las empresas como "Tiker" que se detecto como Unnamed (sin nombre).

In [ ]:
anios = [2014, 2015, 2016, 2017, 2018]
tablas = []

for anio in anios:
    tabla = pd.read_csv(f"{anio}_Financial_Data.csv")
    tabla = tabla.rename(columns={"Unnamed: 0": "Ticker"})
    # la columna de variación de precio se llama '<anio+1> PRICE VAR [%]': la generalizamos
    columna_precio = [c for c in tabla.columns if "PRICE VAR" in c][0]
    tabla = tabla.rename(columns={columna_precio: "Price Var Next Year"})
    tabla["Year"] = anio
    tablas.append(tabla)

df_completo = pd.concat(tablas, ignore_index=True)
print("Filas:", df_completo.shape[0], "| Columnas:", df_completo.shape[1])
df_completo.head()

In [ ]:
#comprobación

print("Dimensiones:", df_completo.shape)


print("\nRegistros por año:\n", df_completo["Year"].value_counts())

print("\nDuplicados por Ticker y Año:", df_completo.duplicated(subset=["Ticker", "Year"]).sum())


print("\nSectores únicos:\n", df_completo["Sector"].unique())

In [ ]:
#guardamos el dataset unificado
df_completo.to_csv("financial_data_unificado.csv", index=False)

## 2. Diagnóstico inicial (sobre el dataset completo)

Las tres preguntas de siempre: cuántas filas y columnas, de qué tipo es cada variable, y qué tan completa está. Lo hacemos sobre las 225 columnas originales, antes de recortar, para tener una foto completa del dataset crudo.

In [ ]:
# Obtengo el nombre de las columnas
df_completo.info()

In [ ]:
# Tipo de datos en cada columna
df_completo.dtypes

##Análisis de Datos Faltantes sobre Base Completa

In [ ]:
# Obtengo la cantidad de datos faltantes en cada columna (NaN)
datos_faltantes = df_completo.isnull().sum()
print("Datos faltantes de cada columna", datos_faltantes)

# Obtengo el maximo valor de datos faltantes y lo guardo en la variable masnan
masnan = df_completo.isnull().sum().max()
print("Maximo valor de datos faltantes masana", masnan)

#  Calculamos el porcentaje de datos faltantes (cant. de datos faltantes/cant. de datos totales * 100)

resultado1 = (masnan / len(df_completo)) * 100

# Imprimo el maximo porcentaje de datos faltantes (porcentaje de falta de datos de la columna con mas NaN). Uso f-strings.

print(f"El porcentaje máximo de filas nulas es igual a {resultado1:.2f}%")

# Asigno el valor resultado1 al porcentaje
porcentaje = resultado1

por_str = str(porcentaje)

type(porcentaje)

type(por_str)

print(type(por_str))


##Análisis Datos Duplicados

In [ ]:
#Analizamos la existencia de filas Duplicadas
print("Filas duplicadas:", df_completo.duplicated().sum())
print("Empresas (tickers) distintas:", df_completo["Ticker"].nunique())
print("Sectores:", df_completo["Sector"].nunique())
df_completo["Sector"].value_counts()

##Análisis de Sectores

In [ ]:
#Contamos la cantidad de empresas por sector y año
tabla = df_completo.groupby(['Sector', 'Year']).size().unstack(fill_value=0)
display(tabla)

In [ ]:
#Columnas
print("lista de columnas",df_completo.columns.tolist())

# Obtengo el promedio de la columna (Revenue-Ingresos Totales-permite calcular cuál fue la facturación promedio de las empresas por año.)

print("La facturación promedio de las empresas por año fue de: ",df_completo.groupby('Year' )["Revenue"].mean())

# Obtengo el promedio por sector y año (.unstack() para que se vizualizar bien la tabla de doble entrada y .display() genera una tabla HTML interactiva)

print("--- Facturación promedio por sector y año ---")
display(df_completo.groupby(['Sector', 'Year'])['Revenue'].mean().unstack())

## 3. Selección de columnas

Del dataset completo (225 columnas) nos quedamos con las que definimos por criterio financiero: valuación, rentabilidad, liquidez, endeudamiento, crecimiento y dividendos, más `Ticker`, `Year` y `Sector` para identificar cada fila.

##De la base de datos completa se utilizaran solamente las siguientes columnas:
**Valuación — "¿está cara o barata la acción?"**

•	**P/E (Price/Earnings):** cuántos años de ganancia actual hacen falta para "pagar" el precio de la acción. Bajo = parece barata (o el mercado no confía en su futuro); alto = cara (o el mercado espera que crezca mucho).

•	**P/B (Price/Book):** precio de la acción vs. el valor contable de sus activos. Sirve para ver si el mercado paga mucho más de lo que la empresa "vale en el papel".

**Rentabilidad — "¿gana plata de forma eficiente?"**

•	**ROE (Return on Equity):** por cada $1 que pusieron los accionistas, cuánta ganancia generó la empresa. Es el ratio más citado en análisis fundamental.

•	**ROA (Return on Assets):** El ROA mide cuánta ganancia genera una empresa por cada peso (o dólar) que tiene invertido en sus activos totales

•	**ROIC (Return of Invested Capital):** El ROIC mide cuánta ganancia operativa genera una empresa por cada peso (o dólar) de capital que realmente se usó para financiar el negocio

•	**Net Profit Margin:** de cada $100 que factura, cuántos terminan siendo ganancia neta (después de todos los gastos e impuestos).

**Liquidez — "¿puede pagar sus deudas de corto plazo?"**

•	**Current Ratio:** activos que puede convertir en efectivo rápido, divididos por lo que debe pagar pronto. Menor a 1 es señal de alerta (podría no cubrir sus obligaciones inmediatas).

•	**Quick Ratio:** indicador financiero que mide la capacidad de una empresa para pagar sus deudas a corto plazo usando solo sus activos más líquidos. Es más estricto y conservador porque excluye los inventarios, ya que estos tardan más tiempo en convertirse en dinero en efectivo.

•	**Free Cash Flow Margin:** % de las ventas que se convierte en caja libre (comparable entre empresas)

**Endeudamiento — "¿cuánta deuda tiene en relación a lo suyo?"**

•	**Debt to Equity:** deuda total / patrimonio de los accionistas. Alto = la empresa se financia mucho con deuda (más riesgo si suben las tasas o caen las ventas).

• **Interest Coverage:** Mide cuántas veces la ganancia operativa cubre los intereses de la deuda.

**Crecimiento — "¿está creciendo o estancada?"**

•	**Revenue Growth (YoY):** cuánto crecieron las ventas respecto al año anterior. Un solo número, no una serie — está permitido.

•	**EPS Growth (YoY):** cuánto creció la ganancia por acción respecto al año anterior.

**Dividendos y contexto**

•	**Dividend Yield:** qué % del precio de la acción se devuelve como dividendo anual.

•	**Market Cap:** tamaño de la empresa (grande/mediana/chica) — importante porque empresas grandes y chicas se comportan distinto.

•	**Sector:** la categórica clave — sin esto no se puede comparar "contra la industria" ni hacer clustering con sentido.

•	**VAR % Año Próximo:** la variación del precio de la acción en el año posterior al de análisis.

In [ ]:
columnas_conservar = [
    # Identificadores
    "Ticker", "Year", "Sector", "Market Cap", "Price Var Next Year", "Class",
    # Valoración (separado del score)
    "PE ratio", "PB ratio",
    # Rentabilidad
    "ROE", "returnOnAssets", "ROIC", "Net Profit Margin", "Free Cash Flow margin",
    # Liquidez
    "Current ratio", "quickRatio",
    # Apalancamiento
    "Debt to Equity", "Interest Coverage",
    # Crecimiento
    "Revenue Growth", "EPS Growth",
    # Dividendos
    "Dividend Yield",
]
# Dropeamos todo lo que no está en columnas_deseadas y nos quedamos con df_completo recortado
df_reducido = df_completo[columnas_conservar].copy()
print("Columnas:", df_reducido.shape[1], "| Filas:", df_reducido.shape[0])
df_reducido.head()

### Traducimos las columnas al español pero mantenemos las siglas del ingles que se usan tambien en español

Recién ahora, con el dataset ya recortado, aplicamos la normalización de nombres (todo en minúscula, sin espacios, con guión bajo como separador) para trabajar más cómodos en el resto de la notebook.

In [ ]:
# #def normalizar_nombre(columna):
#     """Pasa un nombre de columna a minúsculas y reemplaza separadores por guión bajo."""
#     nombre = columna.strip().lower()
#     nombre = re.sub(r"[ /&\-]+", "_", nombre)
#     return nombre

# df_reducido.columns = [normalizar_nombre(c) for c in df_reducido.columns]
# df_reducido.columns.tolist()

In [ ]:
traduccion = {
    "Ticker": "ticker",
    "Year": "anio",
    "Sector": "sector",
    "Market Cap": "capitalizacion_mercado",
    "PE ratio": "pe_ratio",
    "PB ratio": "pb_ratio",
    "ROE": "roe",
    "returnOnAssets": "roa",
    "ROIC": "roic",
    "Net Profit Margin": "margen_neto",
    "Free Cash Flow margin": "margen_flujo_caja_libre",
    "Current ratio": "liquidez_corriente",
    "quickRatio": "liquidez_inmediata",
    "Debt to Equity": "deuda_patrimonio",
    "Interest Coverage": "cobertura_intereses",
    "Revenue Growth": "crecimiento_ingresos",
    "EPS Growth": "crecimiento_eps",
    "Dividend Yield": "rendimiento_dividendo",
    "Price Var Next Year": "variacion_precio_siguiente_anio",
    "Class": "clase",
}

df_reducido = df_reducido.rename(columns=traduccion)
print(df_reducido.columns.tolist())

## 4. Valores faltantes


In [ ]:


faltantes = pd.DataFrame({
    "cantidad": df_reducido.isna().sum(),
    "porcentaje": (df_reducido.isna().mean() * 100).round(2),
})
faltantes[faltantes["cantidad"] > 0].sort_values("porcentaje", ascending=False)



In [ ]:
# Lista de los 15 ratios financieros (excluye identificadores y columnas de validación)
columnas_ratio = [
    "capitalizacion_mercado", "pe_ratio", "pb_ratio", "roe", "roa", "roic",
    "margen_neto", "margen_flujo_caja_libre", "liquidez_corriente", "liquidez_inmediata",
    "deuda_patrimonio", "cobertura_intereses", "crecimiento_ingresos",
    "crecimiento_eps", "rendimiento_dividendo",
]

# Cuántos de esos 15 ratios le faltan a cada fila (empresa-año)
df_reducido["faltantes_en_fila"] = df_reducido[columnas_ratio].isnull().sum(axis=1)

# Distribución: ¿cuántas filas tienen 0 faltantes, 1 faltante, ..., 15 faltantes?
print("=== Distribución de faltantes por fila ===")
print(df_reducido["faltantes_en_fila"].value_counts().sort_index())
faltantes[faltantes["cantidad"] > 0].sort_values("porcentaje", ascending=False)

## ¿Es aleatorio el faltante o se concentra en algún sector?

In [ ]:
print("\n=== Promedio de faltantes por sector ===")
print(df_reducido.groupby("sector")["faltantes_en_fila"].mean().sort_values(ascending=False))

## ¿Se concentra en algún año?

In [ ]:
print("\n=== Promedio de faltantes por año ===")
print(df_reducido.groupby("anio")["faltantes_en_fila"].mean())

## ¿Hay tickers puntuales con faltante total (15 de 15) en todos sus años?

In [ ]:
#¿Hay tickers puntuales con faltante total (15 de 15) en todos sus años?
faltantes_por_ticker = (
    df_reducido.groupby("ticker")["faltantes_en_fila"].mean()
    .sort_values(ascending=False)
)
print("\n=== Top 10 tickers con más faltantes promedio ===")
print(faltantes_por_ticker.head(100))

## Conclusión

Eliminar las filas con los 15 ratios 100% vacíos.
Completar el resto de los faltantes con la mediana del grupo sector + anio

In [ ]:
# Guardamos cuántas filas había antes, para poder reportar el impacto
filas_antes = len(df_reducido)

# Eliminamos las filas donde faltan los 15 ratios (no aportan nada evaluable)
df_reducido = df_reducido[df_reducido["faltantes_en_fila"] < 15].copy()

filas_despues = len(df_reducido)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues,
      f"({(filas_antes - filas_despues) / filas_antes * 100:.1f}% del total)")

In [ ]:
#Completa el resto de los faltantes con la mediana del grupo sector + anio
# Imputación por grupo: mediana dentro de cada combinación sector-año
for col in columnas_ratio:
    df_reducido[col] = df_reducido.groupby(["sector", "anio"])[col].transform(
        lambda serie: serie.fillna(serie.median())
    )

print("Faltantes restantes:", int(df_reducido[columnas_ratio].isnull().sum().sum()))
df_reducido.isnull().sum()

Con esta imputación, cada empresa que tenía un ratio faltante ahora tiene el valor típico de su sector-año en ese ratio específico — es decir, para el score de salud financiera, esa empresa va a quedar "neutral" (ni particularmente sana ni en riesgo) en esa dimensión puntual, en vez de ser excluida. Vale la pena mencionarlo como supuesto metodológico en el informe final.

In [ ]:
# columna auxiliar: no puede entrar al modelo
df_reducido = df_reducido.drop(columns="faltantes_en_fila")

## 5. Outliers


In [ ]:
def winsorizar(serie, percentil_bajo=0.01, percentil_alto=0.99):
    """Recorta una serie a sus percentiles bajo y alto."""
    limite_bajo = serie.quantile(percentil_bajo)
    limite_alto = serie.quantile(percentil_alto)
    return serie.clip(limite_bajo, limite_alto)

# TODO: aplicar winsorizar a cada columna de columnas_ratio (guardar en el mismo df o en columnas nuevas)
# TODO: volver a graficar los boxplots después de winsorizar y comparar contra los de arriba


In [ ]:
# Copia winsorizada (se conserva df_reducido para comparar)
df_winsor = df_reducido.copy()
for col in columnas_ratio:
    df_winsor[col] = winsorizar(df_winsor[col])

# Antes y después, en las variables con valores más extremos
variables = ["cobertura_intereses", "margen_neto", "pe_ratio", "crecimiento_eps"]
fig, ejes = plt.subplots(2, len(variables), figsize=(16, 7))
for j, v in enumerate(variables):
    sns.boxplot(y=df_reducido[v], ax=ejes[0, j], color="#4C72B0")
    ejes[0, j].set_title(f"{v} (original)")
    sns.boxplot(y=df_winsor[v], ax=ejes[1, j], color="#55A868")
    ejes[1, j].set_title(f"{v} (winsorizado)")
    for fila in (0, 1):
        ejes[fila, j].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
df_winsor.to_csv("financial_data_limpio.csv", index=False)

# Parte 2: ARMADO DE MODELOS

## Preparación del Entorno